In [144]:
import numpy as np 
import pandas as pd 
import warnings
warnings.filterwarnings("ignore")

In [145]:
df=pd.read_csv("Data.csv")

# Analysing Data (EDA) and Data Pre Processing

In [146]:
df.head()

,Movie Name,Release Period,Whether Remake,Whether Franchise,Genre,New Actor,New Director,New Music Director,Lead Star,Director,Music Director,Number of Screens,Revenue(INR),Budget(INR)
0,Golden Boys,Normal,No,No,suspense,Yes,No,No,Jeet Goswami,Ravi Varma,Baba Jagirdar,5,5000000,85000
1,Kaccha Limboo,Holiday,No,No,drama,Yes,No,Yes,Karan Bhanushali,Sagar Ballary,Amardeep Nijjer,75,15000000,825000
2,Not A Love Story,Holiday,No,No,thriller,No,No,No,Mahie Gill,Ram Gopal Verma,Sandeep Chowta,525,75000000,56700000
3,Qaidi Band,Holiday,No,No,drama,Yes,No,No,Aadar Jain,Habib Faisal,Amit Trivedi,800,210000000,4500000
4,Chaatwali,Holiday,No,No,adult,Yes,Yes,Yes,Aadil Khan,Aadil Khan,Babloo Ustad,1,1000000,1075000


In [147]:
df.shape

(1698, 14)

In [148]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1698 entries, 0 to 1697
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   Movie Name          1698 non-null   object
 1   Release Period      1698 non-null   object
 2   Whether Remake      1698 non-null   object
 3   Whether Franchise   1698 non-null   object
 4   Genre               1698 non-null   object
 5   New Actor           1698 non-null   object
 6   New Director        1698 non-null   object
 7   New Music Director  1698 non-null   object
 8   Lead Star           1698 non-null   object
 9   Director            1698 non-null   object
 10  Music Director      1698 non-null   object
 11  Number of Screens   1698 non-null   int64 
 12  Revenue(INR)        1698 non-null   int64 
 13  Budget(INR)         1698 non-null   int64 
dtypes: int64(3), object(11)
memory usage: 185.8+ KB


In [149]:
df.isnull().sum()

Movie Name            0
Release Period        0
Whether Remake        0
Whether Franchise     0
Genre                 0
New Actor             0
New Director          0
New Music Director    0
Lead Star             0
Director              0
Music Director        0
Number of Screens     0
Revenue(INR)          0
Budget(INR)           0
dtype: int64

In [150]:
df.columns

Index(['Movie Name', 'Release Period', 'Whether Remake', 'Whether Franchise',
       'Genre', 'New Actor', 'New Director', 'New Music Director', 'Lead Star',
       'Director', 'Music Director', 'Number of Screens', 'Revenue(INR)',
       'Budget(INR)'],
      dtype='object')

In [151]:
df=df[['Movie Name','Genre', 'Lead Star','Director', 'Music Director']]

In [152]:
df.head()

,Movie Name,Genre,Lead Star,Director,Music Director
0,Golden Boys,suspense,Jeet Goswami,Ravi Varma,Baba Jagirdar
1,Kaccha Limboo,drama,Karan Bhanushali,Sagar Ballary,Amardeep Nijjer
2,Not A Love Story,thriller,Mahie Gill,Ram Gopal Verma,Sandeep Chowta
3,Qaidi Band,drama,Aadar Jain,Habib Faisal,Amit Trivedi
4,Chaatwali,adult,Aadil Khan,Aadil Khan,Babloo Ustad


In [153]:
df['Genre']=df['Genre'].apply(lambda x:x.lower())
df['Lead Star']=df['Lead Star'].apply(lambda x:x.lower())
df['Director']=df['Director'].apply(lambda x:x.lower())
df['Music Director']=df['Music Director'].apply(lambda x:x.lower())

In [154]:
df.head()

,Movie Name,Genre,Lead Star,Director,Music Director
0,Golden Boys,suspense,jeet goswami,ravi varma,baba jagirdar
1,Kaccha Limboo,drama,karan bhanushali,sagar ballary,amardeep nijjer
2,Not A Love Story,thriller,mahie gill,ram gopal verma,sandeep chowta
3,Qaidi Band,drama,aadar jain,habib faisal,amit trivedi
4,Chaatwali,adult,aadil khan,aadil khan,babloo ustad


In [155]:
df['Tag']=df['Genre'] + " " + df['Lead Star'] + " " + df['Director'] + " " + df['Music Director']

In [156]:
df=df[['Movie Name','Tag']]

In [157]:
df.head()

,Movie Name,Tag
0,Golden Boys,suspense jeet goswami ravi varma baba jagirdar
1,Kaccha Limboo,drama karan bhanushali sagar ballary amardeep ...
2,Not A Love Story,thriller mahie gill ram gopal verma sandeep ch...
3,Qaidi Band,drama aadar jain habib faisal amit trivedi
4,Chaatwali,adult aadil khan aadil khan babloo ustad


In [158]:
indices=pd.Series(data=df.index,index=df['Movie Name'])
indices.head()

Movie Name
Golden Boys         0
Kaccha Limboo       1
Not A Love Story    2
Qaidi Band          3
Chaatwali           4
dtype: int64

# Making Recommendations by NLP model

In [159]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [160]:
vectorizer=TfidfVectorizer(ngram_range=(1,2),lowercase=True,max_features=10000)
X=vectorizer.fit_transform(df['Tag'])

In [161]:
X

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 21404 stored elements and shape (1698, 8625)>

In [162]:
from sklearn.metrics.pairwise import cosine_similarity

In [163]:
def recommend(movie, n=5):
    if movie not in df['Movie Name'].values:
        return ['Movie not found!!']
    else:
        idx=indices[movie]
        sim_score=cosine_similarity(X[idx],X).flatten()
        sim=sim_score.argsort()[::-1][1:n+1]
        return df['Movie Name'].iloc[sim].tolist()

In [164]:
recommend("Singham",7)

['Bol Bachchan',
 'Singham Returns',
 'Golmaal 3',
 'All The Best - Fun Begins',
 'Golmaal - Fun Unlimited',
 'Sunday',
 'Golmaal Returns']

# Saving files for deploying

In [165]:
import pickle

In [166]:
pickle.dump(vectorizer,open('vectorizer.pkl','wb'))
pickle.dump(X, open('X.pkl','wb'))
pickle.dump(indices,open('indices.pkl','wb'))
df.to_pickle('df.pickle')